# Exchangeability Plotting Notebook
Generate publication-style plots from an exchangeability CSV.

In [ ]:
import os
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
from scripts.plot_exchangeability import _prepare, _aggregate_for_curves, _plot_metric, _plot_train_val, _plot_within_observed_significance

In [ ]:
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', '/n/netscratch/kempner_pehlevan_lab/Lab/ilavie/exchangeability_outputs'))
CSV_OVERRIDE = os.environ.get('EXCHANGEABILITY_CSV', '').strip()
REQUIRED_ANALYSIS_COLUMNS = {
    'width',
    'images_seen',
    'representation',
    'analysis_type',
    'shuffle_id',
    'ks_distance',
    'w1_distance',
}

# KS/W1 curve controls (exposed for quick notebook edits)
KS_W1_ANALYSIS_TYPES = [
    'within_vs_across_real',
    'within_shuffled_vs_across_real',
]
KS_W1_ANALYSIS_LABELS = {
    'within_vs_across_real': 'within-across',
    'within_shuffled_vs_across_real': 'within (shuffled)-across (real)',
}
# Per-width color pairing: full color for within-across, lighter shade for within (shuffled)-across (real)
KS_W1_ANALYSIS_COLOR_ADJUST = {
    'within_vs_across_real': 0.0,
    'within_shuffled_vs_across_real': 0.45,
}
KS_W1_REPRESENTATION_LINESTYLES = {
    'weights': '-',
    'activations': '--',
    'activation': '--',
}
KS_W1_REPRESENTATION_ORDER = ['weights', 'activations', 'activation']

if CSV_OVERRIDE:
    candidates = [Path(CSV_OVERRIDE)]
else:
    preferred = [
        BASE_SAVE_DIR / 'exchangeability_metrics.csv',
        Path('outputs') / 'exchangeability_metrics.csv',
    ]
    all_candidates = list(BASE_SAVE_DIR.glob('exchangeability_metrics*.csv')) + list(Path('outputs').glob('exchangeability_metrics*.csv'))
    fallback = sorted(
        [p for p in all_candidates if '_summary' not in p.stem],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    candidates = []
    seen = set()
    for p in preferred + fallback:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        candidates.append(p)

CSV_PATH = None
skipped = []
for p in candidates:
    if not p.exists():
        continue
    try:
        columns = set(pd.read_csv(p, nrows=0).columns.astype(str))
    except Exception as exc:
        skipped.append((p, f'unreadable header: {exc}'))
        continue
    missing = sorted(REQUIRED_ANALYSIS_COLUMNS - columns)
    if missing:
        skipped.append((p, f'missing columns: {missing}'))
        continue
    CSV_PATH = p
    break

if CSV_PATH is None:
    preview = '\n'.join([f'  - {p}: {reason}' for p, reason in skipped[:8]])
    raise FileNotFoundError(
        'No compatible raw exchangeability metrics CSV found (summary files are not valid for plotting).\n'
        + preview
    )
for p, reason in skipped[:3]:
    print(f'Skipping CSV candidate {p}: {reason}')
OUT_DIR = CSV_PATH.parent / f'plots_{CSV_PATH.stem}_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using CSV_PATH={CSV_PATH}')
print(f'Writing OUT_DIR={OUT_DIR}')
print(f'KS/W1 analysis types: {KS_W1_ANALYSIS_TYPES}')
df = _prepare(pd.read_csv(CSV_PATH))
available_widths = sorted(df['width'].astype(int).unique().tolist())
print(f'Available widths in CSV: {available_widths}')
cmap = plt.get_cmap('tab10')
KS_W1_WIDTH_COLORS = {int(w): cmap(i % cmap.N) for i, w in enumerate(available_widths)}
print(f'KS/W1 width colors: {KS_W1_WIDTH_COLORS}')
curves = _aggregate_for_curves(df)


In [ ]:
fig_ks = _plot_metric(
    curves,
    metric='ks_distance',
    lo='ks_distance_p10',
    hi='ks_distance_p90',
    out_path=str(OUT_DIR / 'ks_distance_vs_images_seen.pdf'),
    title='KS Distance vs Images Seen',
    close=False,
    analysis_types=KS_W1_ANALYSIS_TYPES,
    analysis_labels=KS_W1_ANALYSIS_LABELS,
    width_colors=KS_W1_WIDTH_COLORS,
    analysis_color_adjust=KS_W1_ANALYSIS_COLOR_ADJUST,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
)
display(fig_ks)

fig_w1 = _plot_metric(
    curves,
    metric='w1_distance',
    lo='w1_distance_p10',
    hi='w1_distance_p90',
    out_path=str(OUT_DIR / 'w1_distance_vs_images_seen.pdf'),
    title='W1 Distance vs Images Seen',
    close=False,
    analysis_types=KS_W1_ANALYSIS_TYPES,
    analysis_labels=KS_W1_ANALYSIS_LABELS,
    width_colors=KS_W1_WIDTH_COLORS,
    analysis_color_adjust=KS_W1_ANALYSIS_COLOR_ADJUST,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
)
display(fig_w1)

for fig in _plot_train_val(df, str(OUT_DIR), close=False):
    display(fig)


for fig in _plot_within_observed_significance(
    df,
    str(OUT_DIR),
    close=False,
    width_colors=KS_W1_WIDTH_COLORS,
    representation_linestyles=KS_W1_REPRESENTATION_LINESTYLES,
    representation_order=KS_W1_REPRESENTATION_ORDER,
):
    display(fig)
plt.close('all')
print(f'Wrote plots to {OUT_DIR}')


In [ ]:
import numpy as np
import matplotlib.colors as mcolors
from itertools import combinations
from scripts.analyze_exchangeability import (
    _resolve_width_dirs,
    _list_group_dirs,
    _collect_target_steps,
    _missing_weight_artifacts,
    _extract_weights_from_artifacts,
    _weight_similarity_matrix,
    _collect_member_states,
    _activation_similarity_matrix,
    _build_probe_loader,
)
from src.experiment.exchangeability_utils import (
    build_member_ids,
    extract_across_values,
    extract_within_values,
    ks_w1_stats,
)

ECDF_DATA_DIR = CSV_PATH.with_name(CSV_PATH.stem + '_similarity')
ECDF_RUN_ID = 'exchangeability'
ECDF_RUN_ID_RESOLUTION = 'latest_prefix'
AVAILABLE_WIDTHS = sorted(df['width'].astype(int).unique().tolist())
ECDF_WIDTH = int(AVAILABLE_WIDTHS[0]) if AVAILABLE_WIDTHS else 32
ECDF_REPRESENTATION = 'weights'  # 'weights' or 'activations'
ECDF_MAX_POINTS = 200000
ECDF_STEPS = None  # set list[int] to restrict specific P values
ECDF_RNG_SEED = 0

# Used only when fallback compute is needed and ECDF_REPRESENTATION='activations'
ECDF_PROBE_BATCH_SIZE = 1024
ECDF_PROBE_LOADER_BATCH_SIZE = 128
ECDF_PROBE_SEED = 1234

PAIRWISE_ACROSS_REPRESENTATIONS = ['weights', 'activations']
PAIRWISE_ACROSS_MAX_POINTS = 200000
PAIRWISE_ACROSS_RNG_SEED = 20260310
PAIRWISE_ACROSS_COMPUTE_MISSING = True

print(f'ECDF_DATA_DIR={ECDF_DATA_DIR}')

SINGLE_NET_REPRESENTATION = 'weights'
SINGLE_NET_SUBPART_FRACTIONS = [0.25, 0.5, 0.75]
SINGLE_NET_SUBPART_REPEATS = 32
SINGLE_NET_SUBPART_RNG_SEED = 20260311
SINGLE_NET_MEMBER_INDEX = 0


In [ ]:
def _ecdf_xy(values: np.ndarray):
    xs = np.sort(values)
    ys = np.arange(1, xs.size + 1, dtype=np.float64) / float(xs.size)
    return xs, ys


def _subsample(values: np.ndarray, max_points: int, rng: np.random.Generator):
    if max_points <= 0 or values.size <= max_points:
        return values
    idx = rng.choice(values.size, size=max_points, replace=False)
    return values[idx]


def _lighten(color, amount: float):
    rgba = np.array(mcolors.to_rgba(color), dtype=np.float64)
    white = np.array([1.0, 1.0, 1.0, rgba[3]], dtype=np.float64)
    mixed = (1.0 - amount) * rgba + amount * white
    return tuple(mixed.tolist())


width_dir = ECDF_DATA_DIR / f'width_{ECDF_WIDTH}'
available_steps = []
for step_dir in sorted(width_dir.glob('step_*')):
    try:
        step = int(step_dir.name.split('_')[-1])
    except ValueError:
        continue
    npz_path = step_dir / f'{ECDF_REPRESENTATION}.npz'
    if npz_path.exists():
        available_steps.append(step)

width_mask = df['width'].astype(int) == int(ECDF_WIDTH)
rep_mask = df['representation'].astype(str) == str(ECDF_REPRESENTATION)
csv_steps = sorted({int(s) for s in df.loc[width_mask & rep_mask, 'images_seen'].tolist()})

can_compute = False
group_dirs = []
target_steps = []
try:
    width_dirs, width_sources = _resolve_width_dirs(
        str(BASE_SAVE_DIR),
        ECDF_RUN_ID,
        ECDF_RUN_ID_RESOLUTION,
        requested_widths=[int(ECDF_WIDTH)],
    )
    if ECDF_WIDTH in width_dirs:
        resolved_run_id = width_sources.get(ECDF_WIDTH, '')
        print(f'Fallback compute resolved run_id={resolved_run_id} for width={ECDF_WIDTH}')
        group_dirs = _list_group_dirs(width_dirs[ECDF_WIDTH])
        if group_dirs:
            target_steps = _collect_target_steps(group_dirs)
            can_compute = True
except Exception as exc:
    print(f'Could not resolve run data for fallback computation: {exc}')

target_step_set = set(target_steps)

if ECDF_STEPS is None:
    steps = sorted(set(available_steps) | set(csv_steps) | target_step_set)
else:
    steps = [int(s) for s in ECDF_STEPS]

if not steps:
    raise ValueError(
        f'No ECDF candidate steps for width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}. '
        f'cache_steps={len(available_steps)}, csv_steps={len(csv_steps)}, compute_steps={len(target_steps)}. '
        f'Expected cache under {width_dir}.'
    )

print(
    f'ECDF step sources: cache={len(available_steps)} csv={len(csv_steps)} '
    f'compute={len(target_steps)} selected={len(steps)}'
)

probe_loader = None
ecdf_data = {}
for step in steps:
    npz_path = width_dir / f'step_{int(step)}' / f'{ECDF_REPRESENTATION}.npz'

    if npz_path.exists():
        with np.load(npz_path) as arr:
            within_real = np.asarray(arr['within_real'])
            across_real = np.asarray(arr['across_real'])
    else:
        if (not can_compute) or (int(step) not in target_step_set):
            print(f'Skipping P={step}: cache missing and fallback compute unavailable for this step.')
            continue

        if ECDF_REPRESENTATION == 'weights':
            missing = _missing_weight_artifacts(group_dirs, int(step))
            if missing:
                print(f'Skipping P={step}: missing weight artifacts ({len(missing)} files).')
                continue
            weight_chunks = [_extract_weights_from_artifacts(group_dir, int(step)) for group_dir in group_dirs]
            weights = np.concatenate(weight_chunks, axis=0)
            num_members = int(weights.shape[0])
            sim = _weight_similarity_matrix(weights)
        elif ECDF_REPRESENTATION == 'activations':
            if probe_loader is None:
                probe_loader = _build_probe_loader(
                    probe_batch_size=ECDF_PROBE_BATCH_SIZE,
                    probe_seed=ECDF_PROBE_SEED,
                    probe_loader_batch_size=ECDF_PROBE_LOADER_BATCH_SIZE,
                )
            member_states = _collect_member_states(group_dirs, int(step), progress_label=f'ecdf w{ECDF_WIDTH} p{step}')
            num_members = len(member_states)
            sim = _activation_similarity_matrix(
                member_states,
                ECDF_WIDTH,
                probe_loader,
                progress_label=f'ecdf w{ECDF_WIDTH} p{step}',
            )
        else:
            raise ValueError("ECDF_REPRESENTATION must be 'weights' or 'activations'.")

        member_ids = build_member_ids(num_members, ECDF_WIDTH)
        within_real = extract_within_values(sim, member_ids)
        across_real = extract_across_values(sim, member_ids)

        npz_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            npz_path,
            within_real=np.asarray(within_real, dtype=np.float32),
            across_real=np.asarray(across_real, dtype=np.float32),
            width=np.int32(ECDF_WIDTH),
            images_seen=np.int64(step),
            representation=np.asarray(ECDF_REPRESENTATION),
        )
        print(f'Computed and saved {npz_path}')

    rng = np.random.default_rng(ECDF_RNG_SEED + int(step))
    within_real = _subsample(np.asarray(within_real), ECDF_MAX_POINTS, rng)
    across_real = _subsample(np.asarray(across_real), ECDF_MAX_POINTS, rng)

    ecdf_data[int(step)] = {
        'within_real': within_real,
        'across_real': across_real,
    }

if not ecdf_data:
    raise ValueError(
        f'No ECDF data prepared for width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}. '
        f'Tried {len(steps)} step(s); all were missing cache and could not be recomputed.'
    )

print(f'Prepared real ECDF distributions for {len(ecdf_data)} P values.')


In [ ]:
if not ecdf_data:
    raise ValueError('No ECDF data was prepared.')

steps_sorted = sorted(ecdf_data.keys())
fig, ax = plt.subplots(figsize=(12, 8))
cmap = plt.get_cmap('viridis')
den = max(len(steps_sorted) - 1, 1)

for idx, step in enumerate(steps_sorted):
    base_color = cmap(idx / den)
    pair_color = _lighten(base_color, amount=0.35)

    x_wr, y_wr = _ecdf_xy(ecdf_data[step]['within_real'])
    x_ar, y_ar = _ecdf_xy(ecdf_data[step]['across_real'])

    ax.plot(x_wr, y_wr, color=base_color, linewidth=2.0, linestyle='-', label=f'P={step} within_real')
    ax.plot(x_ar, y_ar, color=pair_color, linewidth=1.7, linestyle='-', label=f'P={step} across_real')

ax.set_xlabel('Absolute cosine similarity')
ax.set_ylabel('ECDF')
ax.set_title(f'ECDF by P (width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION}, real only)')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()

ecdf_out = OUT_DIR / f'ecdf_similarity_real_by_p_w{ECDF_WIDTH}_{ECDF_REPRESENTATION}.pdf'
fig.savefig(ecdf_out, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Wrote {ecdf_out}')


In [ ]:
pairwise_state = {'probe_loader': None}


def _prepare_across_real_by_step(width: int, representation: str):
    width = int(width)
    representation = str(representation)

    width_dir = ECDF_DATA_DIR / f'width_{width}'
    available_steps = []
    for step_dir in sorted(width_dir.glob('step_*')):
        try:
            step = int(step_dir.name.split('_')[-1])
        except ValueError:
            continue
        if (step_dir / f'{representation}.npz').exists():
            available_steps.append(step)

    width_mask = df['width'].astype(int) == width
    rep_mask = df['representation'].astype(str) == representation
    csv_steps = sorted({int(s) for s in df.loc[width_mask & rep_mask, 'images_seen'].tolist()})

    can_compute = False
    group_dirs = []
    target_steps = []
    if PAIRWISE_ACROSS_COMPUTE_MISSING:
        try:
            width_dirs, width_sources = _resolve_width_dirs(
                str(BASE_SAVE_DIR),
                ECDF_RUN_ID,
                ECDF_RUN_ID_RESOLUTION,
                requested_widths=[width],
            )
            if width in width_dirs:
                resolved_run_id = width_sources.get(width, '')
                print(
                    f'Pairwise fallback resolved run_id={resolved_run_id} '
                    f'for width={width}, representation={representation}'
                )
                group_dirs = _list_group_dirs(width_dirs[width])
                if group_dirs:
                    target_steps = _collect_target_steps(group_dirs)
                    can_compute = True
        except Exception as exc:
            print(
                f'Could not resolve run data for width={width}, '
                f'representation={representation}: {exc}'
            )

    target_step_set = set(target_steps)
    if ECDF_STEPS is None:
        steps = sorted(set(available_steps) | set(csv_steps) | target_step_set)
    else:
        steps = [int(s) for s in ECDF_STEPS]

    across_by_step = {}
    loaded_count = 0
    computed_count = 0
    skipped_count = 0

    for step in steps:
        npz_path = width_dir / f'step_{int(step)}' / f'{representation}.npz'

        if npz_path.exists():
            with np.load(npz_path) as arr:
                across_real = np.asarray(arr['across_real'])
            loaded_count += 1
        else:
            if (not can_compute) or (int(step) not in target_step_set):
                skipped_count += 1
                continue

            if representation == 'weights':
                missing = _missing_weight_artifacts(group_dirs, int(step))
                if missing:
                    skipped_count += 1
                    continue
                weight_chunks = [_extract_weights_from_artifacts(group_dir, int(step)) for group_dir in group_dirs]
                weights = np.concatenate(weight_chunks, axis=0)
                num_members = int(weights.shape[0])
                sim = _weight_similarity_matrix(weights)
            elif representation == 'activations':
                if pairwise_state['probe_loader'] is None:
                    pairwise_state['probe_loader'] = _build_probe_loader(
                        probe_batch_size=ECDF_PROBE_BATCH_SIZE,
                        probe_seed=ECDF_PROBE_SEED,
                        probe_loader_batch_size=ECDF_PROBE_LOADER_BATCH_SIZE,
                    )
                member_states = _collect_member_states(
                    group_dirs,
                    int(step),
                    progress_label=f'pairwise w{width} p{step}',
                )
                num_members = len(member_states)
                sim = _activation_similarity_matrix(
                    member_states,
                    width,
                    pairwise_state['probe_loader'],
                    progress_label=f'pairwise w{width} p{step}',
                )
            else:
                raise ValueError("Representation must be 'weights' or 'activations'.")

            member_ids = build_member_ids(num_members, width)
            within_real = extract_within_values(sim, member_ids)
            across_real = extract_across_values(sim, member_ids)

            npz_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(
                npz_path,
                within_real=np.asarray(within_real, dtype=np.float32),
                across_real=np.asarray(across_real, dtype=np.float32),
                width=np.int32(width),
                images_seen=np.int64(step),
                representation=np.asarray(representation),
            )
            computed_count += 1

        across_by_step[int(step)] = np.asarray(across_real)

    meta = {
        'available_steps': len(available_steps),
        'csv_steps': len(csv_steps),
        'target_steps': len(target_steps),
        'selected_steps': len(steps),
        'loaded': loaded_count,
        'computed': computed_count,
        'skipped': skipped_count,
    }
    return across_by_step, meta


if len(AVAILABLE_WIDTHS) < 2:
    raise ValueError('Need at least two widths for pairwise across-distribution W1.')

width_pairs = list(combinations(AVAILABLE_WIDTHS, 2))
selected_steps = set(int(s) for s in ECDF_STEPS) if ECDF_STEPS is not None else None

fig, axes = plt.subplots(
    1,
    len(PAIRWISE_ACROSS_REPRESENTATIONS),
    figsize=(9 * len(PAIRWISE_ACROSS_REPRESENTATIONS), 6),
    squeeze=False,
)
axes = axes[0]

for ax, representation in zip(axes, PAIRWISE_ACROSS_REPRESENTATIONS):
    across_by_width = {}
    meta_by_width = {}

    for width in AVAILABLE_WIDTHS:
        prepared, meta = _prepare_across_real_by_step(int(width), representation)
        across_by_width[int(width)] = prepared
        meta_by_width[int(width)] = meta
        print(
            f'pairwise availability rep={representation} width={int(width)}: '
            f'steps={len(prepared)} loaded={meta["loaded"]} '
            f'computed={meta["computed"]} skipped={meta["skipped"]}'
        )

    plotted_pairs = 0
    for width_a, width_b in width_pairs:
        steps_common = sorted(set(across_by_width[int(width_a)]) & set(across_by_width[int(width_b)]))
        if selected_steps is not None:
            steps_common = [step for step in steps_common if step in selected_steps]
        if not steps_common:
            continue

        w1_values = []
        for step in steps_common:
            rng = np.random.default_rng(
                PAIRWISE_ACROSS_RNG_SEED + int(step) + 1000 * int(width_a) + int(width_b)
            )
            sample_a = _subsample(
                np.asarray(across_by_width[int(width_a)][step]),
                PAIRWISE_ACROSS_MAX_POINTS,
                rng,
            )
            sample_b = _subsample(
                np.asarray(across_by_width[int(width_b)][step]),
                PAIRWISE_ACROSS_MAX_POINTS,
                rng,
            )
            w1_values.append(float(ks_w1_stats(sample_a, sample_b)['w1_distance']))

        if not w1_values:
            continue

        plotted_pairs += 1
        ax.plot(
            steps_common,
            w1_values,
            marker='o',
            linewidth=1.6,
            label=f'N={int(width_a)} vs N={int(width_b)}',
        )

    ax.set_xscale('log')
    ax.set_xlabel('Images seen (P)')
    ax.set_ylabel('W1 distance between across_real distributions')
    ax.set_title(f'Across-width W1 vs P ({representation})')
    ax.grid(True, alpha=0.25)

    if plotted_pairs:
        ax.legend(fontsize=7, ncol=2)
    else:
        ax.text(
            0.5,
            0.5,
            'No width-pair overlap found in cached or computed data.',
            ha='center',
            va='center',
            transform=ax.transAxes,
        )

    print(
        f'pairwise plotted rep={representation}: {plotted_pairs}/{len(width_pairs)} '
        f'width-pair lines'
    )

fig.tight_layout()

pairwise_out = OUT_DIR / 'w1_across_width_pairs_vs_images_seen.pdf'
fig.savefig(pairwise_out, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Wrote {pairwise_out}')


In [ ]:
if SINGLE_NET_REPRESENTATION != 'weights':
    raise ValueError("SINGLE_NET_REPRESENTATION must be 'weights' for this plot.")

width_dirs, width_sources = _resolve_width_dirs(
    str(BASE_SAVE_DIR),
    ECDF_RUN_ID,
    ECDF_RUN_ID_RESOLUTION,
    requested_widths=[int(ECDF_WIDTH)],
)
if int(ECDF_WIDTH) not in width_dirs:
    raise ValueError(f'Could not resolve run directory for width={ECDF_WIDTH}.')

resolved_run_id = width_sources.get(int(ECDF_WIDTH), '')
width_dir = width_dirs[int(ECDF_WIDTH)]
group_dirs = _list_group_dirs(width_dir)
if not group_dirs:
    raise ValueError(f'No group directories found under {width_dir}.')

selected_group_dir = group_dirs[0]
print(
    f'Single-network subpart W1: width={ECDF_WIDTH} run_id={resolved_run_id} '
    f'group={Path(selected_group_dir).name} member_index={SINGLE_NET_MEMBER_INDEX}'
)

artifacts_dir = Path(selected_group_dir) / 'artifacts'
available_steps = []
if artifacts_dir.exists():
    for artifact_path in sorted(artifacts_dir.glob('first_layer_*.npz')):
        try:
            step = int(artifact_path.stem.split('_')[-1])
        except ValueError:
            continue
        available_steps.append(step)
available_steps = sorted(set(available_steps))

if ECDF_STEPS is None:
    steps = available_steps
else:
    steps = [int(s) for s in ECDF_STEPS]

if not steps:
    raise ValueError(
        f'No candidate steps for single-network subpart W1: '
        f'ECDF_STEPS={ECDF_STEPS}, available_artifact_steps={len(available_steps)}.'
    )

fractions = list(dict.fromkeys(float(f) for f in SINGLE_NET_SUBPART_FRACTIONS))
if not fractions:
    raise ValueError('SINGLE_NET_SUBPART_FRACTIONS must be non-empty.')

results = {
    frac: {
        'steps': [],
        'w1_mean': [],
        'w1_p10': [],
        'w1_p90': [],
        'subpart_width': None,
    }
    for frac in fractions
}

processed_steps = 0
for step in steps:
    artifact_path = artifacts_dir / f'first_layer_{int(step)}.npz'
    if not artifact_path.exists():
        print(f'Skipping P={step}: missing weight artifact {artifact_path}.')
        continue

    weights = _extract_weights_from_artifacts(selected_group_dir, int(step))
    if weights.ndim < 2:
        print(f'Skipping P={step}: unexpected weight shape {weights.shape}.')
        continue

    num_members = int(weights.shape[0])
    width_channels = int(weights.shape[1])
    member_idx = int(SINGLE_NET_MEMBER_INDEX)
    if member_idx < 0 or member_idx >= num_members:
        print(
            f'Skipping P={step}: member index {member_idx} out of range for '
            f'num_members={num_members}.'
        )
        continue

    member_weights = np.asarray(weights[member_idx:member_idx + 1], dtype=np.float32)
    sim_full = _weight_similarity_matrix(member_weights)
    tri_full = np.triu_indices(sim_full.shape[0], k=1)
    full_values = np.asarray(sim_full[tri_full], dtype=np.float64)
    if full_values.size == 0:
        print(f'Skipping P={step}: full distribution is empty.')
        continue

    for frac_idx, frac in enumerate(fractions):
        subpart_width = int(round(float(frac) * width_channels))
        subpart_width = max(2, min(width_channels, subpart_width))

        rng = np.random.default_rng(
            int(SINGLE_NET_SUBPART_RNG_SEED)
            + int(step) * 1000
            + frac_idx * 1000000
        )

        w1_samples = []
        for _ in range(int(SINGLE_NET_SUBPART_REPEATS)):
            subset_idx = rng.choice(width_channels, size=subpart_width, replace=False)
            sub_sim = sim_full[np.ix_(subset_idx, subset_idx)]
            tri_sub = np.triu_indices(subpart_width, k=1)
            sub_values = np.asarray(sub_sim[tri_sub], dtype=np.float64)
            if sub_values.size == 0:
                continue
            w1 = float(ks_w1_stats(sub_values, full_values)['w1_distance'])
            w1_samples.append(w1)

        if not w1_samples:
            continue

        stats = np.asarray(w1_samples, dtype=np.float64)
        entry = results[frac]
        entry['steps'].append(int(step))
        entry['w1_mean'].append(float(np.mean(stats)))
        entry['w1_p10'].append(float(np.percentile(stats, 10)))
        entry['w1_p90'].append(float(np.percentile(stats, 90)))
        entry['subpart_width'] = int(subpart_width)

    processed_steps += 1

if processed_steps == 0:
    raise ValueError('No steps were processed for single-network subpart-vs-full W1 plot.')

plotted_fractions = [
    frac for frac in fractions if len(results[frac]['steps']) > 0
]
if not plotted_fractions:
    raise ValueError('No W1 values were produced for any subpart fraction.')

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.get_cmap('viridis')
den = max(len(plotted_fractions) - 1, 1)

for idx, frac in enumerate(plotted_fractions):
    row = results[frac]
    order = np.argsort(np.asarray(row['steps'], dtype=np.int64))
    xs = np.asarray(row['steps'], dtype=np.int64)[order]
    ys = np.asarray(row['w1_mean'], dtype=np.float64)[order]
    lo = np.asarray(row['w1_p10'], dtype=np.float64)[order]
    hi = np.asarray(row['w1_p90'], dtype=np.float64)[order]

    color = cmap(idx / den)
    label = f'r={frac:g} ({int(row["subpart_width"])} channels)'
    ax.plot(xs, ys, marker='o', linewidth=1.7, color=color, label=label)
    ax.fill_between(xs, lo, hi, color=color, alpha=0.15)

ax.set_xscale('log')
ax.set_xlabel('Images seen (P)')
ax.set_ylabel('W1 distance (subpart vs full similarity distributions)')
ax.set_title(
    f'Single-network subpart-vs-full W1 '
    f'(width={ECDF_WIDTH}, rep={SINGLE_NET_REPRESENTATION}, member={SINGLE_NET_MEMBER_INDEX})'
)
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()

single_net_out = OUT_DIR / f'w1_single_network_subparts_vs_full_w{ECDF_WIDTH}_{SINGLE_NET_REPRESENTATION}.pdf'
fig.savefig(single_net_out, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Wrote {single_net_out}')
